In [1]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.optim import Adam
from torchvision import datasets, transforms


In [2]:
print('==> Preparing data.............................')
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, SVHN

import os
import torch
from torch import nn,optim
import torch.nn.functional as F

from torchvision import datasets, transforms

from time import perf_counter

import  numpy as np
import torch.utils.data as Data

from torch.utils.data import Dataset, DataLoader

class Safeman(Dataset):
    
    def __init__(self, data,targets):
        super(Safeman, self).__init__()
        self.data = data
        self.targets = targets
        
     
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        img, target = self.data[idx], self.targets[idx]
        return img, target

class Safeman_Filter(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)
        
class Safeman_FilterB(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        new_targets = []
        for i in range(len(targets)):
            if targets[i] in known:
                new_targets.append(0)
            else:
                new_targets.append(1)
        self.targets = np.array(new_targets)
        self.data = self.data

class Safeman_FilterC(Safeman):
    
    def __Filter__(self, trainknown):
        train_class_num=len(trainknown)
        for i in range(0,len(self.targets)) :
            if self.targets[i]>train_class_num:
                self.targets[i] = train_class_num
        self.data = self.data

        
        
class Safeman_FilterF(Safeman):

    def __Filter__(self, known):
        targets = self.targets.data.numpy()
        mask, new_targets = [], []
        for i in range(len(targets)):
            if targets[i] in known:
                mask.append(i)
                dd = known.index(targets[i])
                if dd == 4:
                    new_targets.append(0)
                else:
                    new_targets.append(1)                   
                    
                #new_targets.append(known.index(targets[i]))
        self.targets = np.array(new_targets)
        mask = torch.tensor(mask).long()
        self.data = torch.index_select(self.data, 0, mask)   


        
def setup_seed(seed):

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(8)



known=[0, 1, 2,3,4,5,6,7]
unknown=[ 5,6,7]





X_train0 = np.load('./TONdataset/x_train_iot1028+1del.npy')
y_train1 = np.load('./TONdataset/y_train_iot1028+1del.npy')
X_final_test0 = np.load('./TONdataset/x_test_iot1028+1del.npy' )
y__final_test1 = np.load('./TONdataset/y_test_iot1028+1del.npy')



X_train1=[]
X_final_test1=[]

for i in range(len(y_train1)):
    #a = np.resize(X_train0[i], (3, 32, 32))
    a = np.resize(X_train0[i], (1, 28, 28))
    X_train1 += [a]
    
for j in range(len(y__final_test1)):
    #b = np.resize(X_final_test0[j], (3, 32, 32))
    b = np.resize(X_final_test0[j], (1, 28, 28))
    X_final_test1 += [b]

i=0
j=0



x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)

print(x_train.shape, x_test.shape, y_train.shape,y_test.shape)

train_dataset = Data.TensorDataset(x_train, y_train)
train_dataset.data = train_dataset.tensors[0]
train_dataset.targets = train_dataset.tensors[1]



test_dataset = Data.TensorDataset(x_test, y_test)
test_dataset.data = test_dataset.tensors[0]
test_dataset.targets = test_dataset.tensors[1]


labels =['backdoor', 'ddos', 'dos', 'injection', 'normal', 'password', 'scanning', 'xss']

train_dataset.classes = labels
test_dataset.classes = labels

train_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}
test_dataset.classes_to_idx = {i: label for i, label in enumerate(labels)}

num_class=len(labels)

b_s=256


trainset = Safeman_Filter(data=train_dataset.data,targets=train_dataset.targets)
print('All down Train Data:', len(trainset))
trainset.__Filter__(known=known)


train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data:', len(trainset))



testsetA = Safeman_Filter(data=test_dataset.data,targets=test_dataset.targets)
print('All testsetA Data:', len(testsetA))
testsetA.__Filter__(known=known)


test_loader_A = torch.utils.data.DataLoader(
    testsetA, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real testsetA Data:', len(testsetA))


print("done!")

==> Preparing data.............................


/tmp/ipykernel_1807288/1189467501.py:150: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  x_train, x_test, y_train,y_test = torch.Tensor(X_train1), torch.Tensor(X_final_test1), torch.from_numpy(y_train1), torch.from_numpy(y__final_test1)


torch.Size([242000, 1, 28, 28]) torch.Size([48000, 1, 28, 28]) torch.Size([242000]) torch.Size([48000])
All down Train Data: 242000
Real train Data: 242000
All testsetA Data: 48000
Real testsetA Data: 48000
done!


In [3]:
unique,counts = np.unique(trainset.targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [4]:
X_trainset_data=trainset.data
X_trainset_targets=trainset.targets

In [5]:
unique,counts = np.unique(X_trainset_targets,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [ 16000  16000  16000  16000 130000  16000  16000  16000]


In [6]:
X_trainset_targets

array([6, 6, 6, ..., 0, 0, 0], shape=(242000,))

In [7]:
count=[10, 10, 10, 10, 130000, 10, 10, 10]
num_class=8
lists = [[] for i in range(num_class)]
y_train_temp=[]
x_train_temp=[]

In [8]:
for i in range(len(X_trainset_targets)):
    if len(lists[X_trainset_targets[i]])<count[X_trainset_targets[i]]:
        lists[X_trainset_targets[i]].append(X_trainset_targets[i])   
        y_train_temp+=[X_trainset_targets[i]]
        a = np.resize(X_trainset_data[i], (1, 28, 28))
        x_train_temp += [a]


In [9]:
unique,counts = np.unique(y_train_temp,return_counts=True)
print(unique, counts)

[0 1 2 3 4 5 6 7] [    10     10     10     10 130000     10     10     10]


In [10]:
x_train_temp[1]

array([[[0.0000000e+00, 4.0047044e-01, 6.5473884e-01, 3.0163100e-01,
         2.5041966e-02, 5.0000000e-01, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 5.0000000e-01, 0.0000000e+00, 1.3197836e-05,
         9.2891190e-07, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 4.0047044e-01, 6.5473884e-01,
         3.0163100e-01, 2.5041966e-02, 5.0000000e-01, 0.0000000e+00],
        [0.0000000e+00, 0.0000000e+00, 5.0000000e-01, 0.0000000e+00,
         1.3197836e-05, 9.2891190e-07, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 4.0047044e-01,
         6.5473884e-01, 3.0163100e-01, 2.5041966e-02, 5.0000000e-01,
         0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 5.0000000e-01,
         0.0000000e+00, 1.3197836e-05, 9.2891190e-07, 0.0000000e+00],
        [0.0000000e+00, 0.000000

In [11]:
x_train2, y_train2 = torch.Tensor(x_train_temp), torch.Tensor(y_train_temp)

print(x_train2.shape, y_train2.shape)

train_dataset2 = Data.TensorDataset(x_train2, y_train2)
train_dataset2.data = train_dataset2.tensors[0]
train_dataset2.targets = train_dataset2.tensors[1]




trainset2 = Safeman_Filter(data=train_dataset2.data,targets=train_dataset2.targets)
print('All down Train Data:', len(trainset2))
trainset2.__Filter__(known=known)



train_loader2 = torch.utils.data.DataLoader(
    trainset2, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(trainset2))

torch.Size([130070, 1, 28, 28]) torch.Size([130070])
All down Train Data: 130070
Real train Data2: 130070


In [12]:

x_train_aug_320 = np.load("./data/BorderlineSMOTEmal46-iot-multi-sa2-f.npy")

x_train_aug_31_label0 = np.load("./data/BorderlineSMOTEmal46-iot-multi-label-sa2-f.npy")

x_train_aug_32101 = np.load("./ACGANtest/data/acgan-iot-data-f.npy")

x_train_aug_31_label1 = np.load("./ACGANtest/data/acgan-iot-label-f.npy")


x_train_aug_3111=[]
for j in range(len(x_train_aug_32101)):
    b = np.resize(x_train_aug_32101[j], (21))
    x_train_aug_3111 += [b]

x_train_aug_321 =np.array(x_train_aug_3111)

print(x_train_aug_321.shape)




x_train_aug_32 = np.concatenate((x_train_aug_320, x_train_aug_321)) 

x_train_aug_31_label = np.concatenate((x_train_aug_31_label0, x_train_aug_31_label1)) 

(70000, 21)


In [13]:
x_train_aug_32.shape

(118030, 21)

In [14]:
x_train_aug_31=[]
for j in range(len(x_train_aug_32)):
    b = np.resize(x_train_aug_32[j], (1, 28, 28))
    x_train_aug_31 += [b]

In [15]:
x_train_aug_31=np.array(x_train_aug_31)

In [16]:
x_train_aug_31.shape

(118030, 1, 28, 28)

In [17]:
print("trainset2.data.numpy()",trainset2.data.numpy().shape)

x_train_temp1 = np.concatenate((trainset2.data.numpy(), x_train_aug_31)) 

y_train_temp1 = np.concatenate((trainset2.targets, x_train_aug_31_label)) 


print("x_train_temp1",x_train_temp1.shape)

print("y_train_temp1",y_train_temp1.shape)

trainset2.data.numpy() (130070, 1, 28, 28)
x_train_temp1 (248100, 1, 28, 28)
y_train_temp1 (248100,)


In [18]:
x_train21, y_train21 = torch.Tensor(x_train_temp1), torch.Tensor(y_train_temp1)

print(x_train21.shape, y_train21.shape)

train_dataset21 = Data.TensorDataset(x_train21, y_train21)
train_dataset21.data = train_dataset21.tensors[0]
train_dataset21.targets = train_dataset21.tensors[1]


train_loader21 = torch.utils.data.DataLoader(
    train_dataset21, batch_size=b_s, shuffle=True,
    num_workers=4,drop_last=True)


print('Real train Data2:', len(train_dataset21))

torch.Size([248100, 1, 28, 28]) torch.Size([248100])
Real train Data2: 248100


In [19]:
from torch.nn import Module
from torch import nn
import numpy as np
import torch
from torchvision.datasets import mnist
from torch.nn import CrossEntropyLoss
from torch.optim import SGD
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor

num_classB=8


class Modelnsl8(nn.Module):
    def __init__(self):
        # Initialize the nn.Module class
        super().__init__()
        self.fc1 = nn.Linear(28*28, 64)  
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 64)
        self.fc4 = nn.Linear(64, num_classB) 
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return F.log_softmax(x, dim = 1)

In [20]:
test_loader2 = test_loader_A
train_loader2 = train_loader21
model = Modelnsl8()
sgd = SGD(model.parameters(), lr=1e-1)
loss_fn = CrossEntropyLoss()
all_epoch = 88

for current_epoch in range(all_epoch):
    model.train()
    for idx, (train_x, train_label) in enumerate(train_loader2):
        sgd.zero_grad()
        predict_y = model(train_x.float().view(-1, 28*28))#modify in DNN network
        loss = loss_fn(predict_y, train_label.long())
        if idx % 1000 == 0:
            print('idx: {}, loss: {}'.format(idx, loss.sum().item()))
        loss.backward()
        sgd.step()
    print("epoch i=",current_epoch)    
    all_correct_num = 0
    all_sample_num = 0
    model.eval()
    for idx, (test_x, test_label) in enumerate(test_loader2):
        predict_y = model(test_x.float().view(-1, 28*28)).detach()
        predict_y = np.argmax(predict_y, axis=-1)
        current_correct_num = predict_y == test_label
        all_correct_num += np.sum(current_correct_num.numpy(), axis=-1)
        all_sample_num += current_correct_num.shape[0]
    acc = all_correct_num / all_sample_num
    print('accuracy: {:.3f}'.format(acc))
print("training end")


idx: 0, loss: 2.1145665645599365
epoch i= 0
accuracy: 0.528
idx: 0, loss: 0.5565282106399536
epoch i= 1
accuracy: 0.563
idx: 0, loss: 0.5684510469436646
epoch i= 2
accuracy: 0.547
idx: 0, loss: 0.3993142545223236
epoch i= 3
accuracy: 0.576
idx: 0, loss: 0.47955071926116943
epoch i= 4
accuracy: 0.537
idx: 0, loss: 0.438670814037323
epoch i= 5
accuracy: 0.552
idx: 0, loss: 0.4317743480205536
epoch i= 6
accuracy: 0.549
idx: 0, loss: 0.5714938044548035
epoch i= 7
accuracy: 0.536
idx: 0, loss: 0.5587080717086792
epoch i= 8
accuracy: 0.534
idx: 0, loss: 0.3931427299976349
epoch i= 9
accuracy: 0.528
idx: 0, loss: 0.3955100178718567
epoch i= 10
accuracy: 0.533
idx: 0, loss: 0.4416470527648926
epoch i= 11
accuracy: 0.546
idx: 0, loss: 0.529699981212616
epoch i= 12
accuracy: 0.571
idx: 0, loss: 0.4438997507095337
epoch i= 13
accuracy: 0.557
idx: 0, loss: 0.4303918480873108
epoch i= 14
accuracy: 0.653
idx: 0, loss: 0.46944406628608704
epoch i= 15
accuracy: 0.549
idx: 0, loss: 0.35417550802230835
